# 11 · Wrap-up and take-homes / Cierre y ejercicios para casa

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb)

*wrap-up · 5 min + take-homes after the workshop / cierre · 5 min + ejercicios para después*

This final notebook has two jobs:

1. connect the ideas from the whole workshop;
2. let you explore five extensions: **PCA, attention, CP, Cholesky, and audio denoising**.

> 🇪🇸 Este último cuaderno tiene dos objetivos:
>
> 1. conectar las ideas de todo el taller;
> 2. permitirte explorar cinco extensiones: **PCA, atención, CP, Cholesky y reducción de ruido de audio**.

## What you will be able to do / Lo que podrás hacer

- State the one approximation idea connecting pseudoinverse, deconvolution, and Tucker.
- Diagnose the PCA scaling trap on real breast-cancer measurements.
- Build masked attention from two `einsum` contractions.
- Compare CP with Tucker on the same real New York taxi tensor.
- Use Cholesky to turn independent noise into correlated draws and see why covariance changes portfolio risk.
- Denoise a real voice recording with `STFT → truncated SVD → ISTFT` and measure the SNR trade-off.

> 🇪🇸
>
> - Explicar la idea de aproximación que conecta pseudoinversa, deconvolución y Tucker.
> - Detectar el problema de escala de PCA sobre mediciones reales de cáncer de mama.
> - Construir atención enmascarada con dos contracciones `einsum`.
> - Comparar CP con Tucker sobre el mismo tensor real de taxis de Nueva York.
> - Usar Cholesky para transformar ruido independiente en muestras correlacionadas y observar cómo la covarianza cambia el riesgo.
> - Reducir ruido de una grabación real con `STFT → SVD truncada → ISTFT` y medir el compromiso mediante SNR.

## One idea connects the workshop / Una idea conecta todo el taller

Several sections looked very different:

- pseudoinverse solved systems where an ordinary inverse was unavailable;
- deconvolution estimated an image after blur and noise;
- Tucker approximated a tensor with smaller mode-specific factors.

But all three asked the same kind of question:

> **If an exact representation is unavailable, unstable, or unnecessarily expensive, can we build a controlled approximation and measure what we lose?**

> 🇪🇸
>
> **Si una representación exacta no existe, es inestable o cuesta demasiado, ¿podemos construir una aproximación controlada y medir qué perdemos?**

### The workshop habit / El hábito del taller

For almost every tensor problem:

**Meaning → Shape → Operation → Result → Interpretation**

> 🇪🇸
>
> **Significado → Forma → Operación → Resultado → Interpretación**

Never let a correct-looking `.shape` hide the meaning of the axes.

## Setup / Preparación

Run this once.

The live wrap-up only needs the summary above. The code below prepares the five take-home explorations.

> 🇪🇸 Ejecuta esta celda una vez.
>
> El cierre en vivo solo necesita el resumen anterior. El código prepara los cinco ejercicios para casa.

In [ ]:
import hashlib
import io
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import Audio, display
from scipy import signal
from sklearn.datasets import load_breast_cancer

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def snr_db(reference, estimate):
    reference = np.asarray(reference)
    estimate = np.asarray(estimate)
    return 10 * np.log10(
        np.sum(reference**2)
        / np.sum((estimate - reference)**2)
    )

print("EN: Setup ready.")
print("ES: Preparación lista.")

### Interactive workshop map / Mapa interactivo del taller

Choose a topic and recall the main question it answered.

> 🇪🇸 Elige un tema y recuerda la pregunta principal que respondió.

In [ ]:
workshop_topic = widgets.Dropdown(
    options=[
        ("Axes and shape / Ejes y forma", "axes"),
        ("Indexing and broadcasting / Indexación y broadcasting", "indexing"),
        ("Video pipelines / Pipelines de video", "video"),
        ("einsum / Contracción", "einsum"),
        ("Pseudoinverse / Pseudoinversa", "pinv"),
        ("Recursion / Recursión", "recursion"),
        ("Convolution / Convolución", "conv"),
        ("Tucker", "tucker"),
    ],
    value="axes",
    description="Topic / Tema:",
    style={"description_width": "110px"},
)

def explain_workshop_topic(topic):
    messages = {
        "axes": (
            "What does every axis count?",
            "¿Qué cuenta cada eje?",
        ),
        "indexing": (
            "Which observations or features should remain?",
            "¿Qué observaciones o características deben permanecer?",
        ),
        "video": (
            "What temporal information does the pipeline keep or discard?",
            "¿Qué información temporal conserva o descarta el pipeline?",
        ),
        "einsum": (
            "Which indices disappear because they are summed?",
            "¿Qué índices desaparecen porque se suman?",
        ),
        "pinv": (
            "If an ordinary inverse is unavailable, what solution is mathematically appropriate?",
            "Si no existe una inversa ordinaria, ¿qué solución es matemáticamente apropiada?",
        ),
        "recursion": (
            "What information is carried from one step into the next?",
            "¿Qué información pasa de un paso al siguiente?",
        ),
        "conv": (
            "Is this a forward filter, a transposed operator, or a true inverse problem?",
            "¿Es un filtro directo, un operador transpuesto o un verdadero problema inverso?",
        ),
        "tucker": (
            "How much structure should we retain along each semantic mode?",
            "¿Cuánta estructura debemos conservar en cada modo semántico?",
        ),
    }

    en, es = messages[topic]
    print("EN:", en)
    print("ES:", es)

map_output = widgets.interactive_output(
    explain_workshop_topic,
    {"topic": workshop_topic},
)

display(widgets.VBox([workshop_topic, map_output]))

## Take-home A — PCA: the scaling trap / PCA: la trampa de la escala

The Wisconsin Diagnostic Breast Cancer dataset has **30 real measurements per sample**.

But those features use very different numerical units and scales.

PCA asks:

> **Which directions contain the most variance?**

It does **not** ask:

> “Which features are scientifically most important?”

So if one feature is measured with numerically huge values, it can dominate the variance calculation.

### Analogy / Analogía

Imagine comparing:

- height in **meters**;
- salary in **dollars**.

The dollar numbers may be thousands of times larger. PCA can react to those numerical units unless the features are standardized.

> 🇪🇸 PCA busca direcciones de **máxima varianza**, no “importancia científica”.
>
> Si las variables están en escalas muy diferentes, las unidades pueden dominar el resultado.

In [ ]:
# TODO A / TAREA A
#
# EN:
# 1. Load the breast-cancer data.
# 2. Center X and compute SVD.
# 3. How many components explain 95% of variance?
# 4. Inspect the feature variances.
# 5. Standardize every feature and repeat.
# 6. Compare the two cumulative-variance curves.
#
# ES:
# 1. Carga los datos de cáncer de mama.
# 2. Centra X y calcula SVD.
# 3. ¿Cuántos componentes explican el 95% de la varianza?
# 4. Inspecciona las varianzas de las características.
# 5. Estandariza cada característica y repite.
# 6. Compara las dos curvas de varianza acumulada.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

bc = load_breast_cancer()
X_pca = bc.data.astype(float)
y_pca = bc.target
feature_names_pca = list(bc.feature_names)

X_centered = X_pca - X_pca.mean(axis=0)

_, s_raw, Vt_raw = np.linalg.svd(
    X_centered,
    full_matrices=False,
)

frac_raw = s_raw**2 / np.sum(s_raw**2)
cum_raw = np.cumsum(frac_raw)
n95_raw = int(np.argmax(cum_raw >= 0.95) + 1)

feature_var = X_pca.var(axis=0)

X_standardized = (
    X_pca - X_pca.mean(axis=0)
) / X_pca.std(axis=0)

_, s_std, Vt_std = np.linalg.svd(
    X_standardized,
    full_matrices=False,
)

frac_std = s_std**2 / np.sum(s_std**2)
cum_std = np.cumsum(frac_std)
n95_std = int(np.argmax(cum_std >= 0.95) + 1)

print("95% components — raw / sin estandarizar:", n95_raw)
print("95% components — standardized / estandarizado:", n95_std)
print(
    "Feature variance range / Rango de varianzas:",
    f"{feature_var.min():.3e}",
    "→",
    f"{feature_var.max():.3e}",
)
print()
print("EN: PCA is mathematically correct in both cases, but the scientific question changes when scale dominates.")
print("ES: PCA es matemáticamente correcto en ambos casos, pero la pregunta científica cambia cuando domina la escala.")

### Interactive PCA scaling explorer / Explorador interactivo de escala en PCA

Switch between **Raw / Sin estandarizar** and **Standardized / Estandarizado**.

You will see:

- cumulative explained variance;
- number of components needed for 95%;
- a two-component projection.

> 🇪🇸 Cambia entre datos sin estandarizar y estandarizados. Observa la varianza acumulada, cuántos componentes se necesitan para 95% y la proyección en dos componentes.

In [ ]:
pca_mode = widgets.ToggleButtons(
    options=[
        ("Raw / Sin estandarizar", "raw"),
        ("Standardized / Estandarizado", "std"),
    ],
    value="raw",
    description="PCA:",
)

def explore_pca_scaling(mode):
    if mode == "raw":
        matrix = X_centered
        frac = frac_raw
        Vt = Vt_raw
        n95 = n95_raw
        name = "Raw / Sin estandarizar"
    else:
        matrix = X_standardized
        frac = frac_std
        Vt = Vt_std
        n95 = n95_std
        name = "Standardized / Estandarizado"

    scores_2d = matrix @ Vt[:2].T

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 4),
        constrained_layout=True,
    )

    axes[0].plot(
        range(1, len(frac) + 1),
        np.cumsum(frac),
        marker="o",
        markersize=3,
    )
    axes[0].axhline(
        0.95,
        linestyle="--",
        linewidth=1,
    )
    axes[0].axvline(
        n95,
        linestyle="--",
        linewidth=1,
    )
    axes[0].set_xlabel("components / componentes")
    axes[0].set_ylabel("cumulative variance / varianza acumulada")
    axes[0].set_title(
        f"{name}\n95% → {n95} component(s) / componente(s)"
    )

    axes[1].scatter(
        scores_2d[:, 0],
        scores_2d[:, 1],
        c=y_pca,
        s=12,
        alpha=0.7,
    )
    axes[1].set_xlabel("component 1 / componente 1")
    axes[1].set_ylabel("component 2 / componente 2")
    axes[1].set_title(
        f"Two-component view\nVista de dos componentes"
    )

    plt.show()

    print("Mode / Modo:", name)
    print("Components for 95% / Componentes para 95%:", n95)
    print("EN: scaling changes which numerical directions count as high variance.")
    print("ES: la escala cambia qué direcciones numéricas se consideran de alta varianza.")

pca_output = widgets.interactive_output(
    explore_pca_scaling,
    {"mode": pca_mode},
)

display(widgets.VBox([pca_mode, pca_output]))

## Take-home B — Attention is two contractions / La atención son dos contracciones

For this exercise, `Q`, `K`, and `V` are **synthetic by design**.

That is appropriate because we want to isolate the tensor mechanics without adding tokenization or a trained language model.

### Step 1 — compare queries with keys

`scores = einsum("bid,bjd->bij", Q, K)`

`d` disappears.

Output:

`(batch, query_position, key_position)`

### Step 2 — turn scores into weights

Apply softmax across the key-position axis.

Each row becomes a probability-like distribution that sums to `1`.

### Step 3 — combine values

`output = einsum("bij,bjd->bid", weights, V)`

`j` disappears.

### Padding mask / Máscara de padding

A padded key position should receive **zero attention weight**.

So the mask must be applied **before softmax**.

> 🇪🇸 La atención puede entenderse como dos contracciones:
>
> 1. comparar consultas con claves;
> 2. usar esos pesos para combinar valores.
>
> Las posiciones de padding deben recibir peso cero.

In [ ]:
# TODO B / TAREA B
#
# EN:
# 1. Create Q, K, V with shape (4, 12, 16).
# 2. Compute scaled dot-product scores with einsum.
# 3. Apply softmax over key positions.
# 4. Contract weights with V.
# 5. Mask the final 3 key positions BEFORE softmax.
# 6. Verify masked positions receive zero weight.
#
# ES:
# 1. Crea Q, K y V con forma (4,12,16).
# 2. Calcula los puntajes escalados con einsum.
# 3. Aplica softmax sobre posiciones key.
# 4. Contrae los pesos con V.
# 5. Enmascara las últimas 3 posiciones ANTES de softmax.
# 6. Verifica que las posiciones enmascaradas reciban peso cero.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

rng_attention = np.random.default_rng(6)

batch_att, seq_len, dim_att = 4, 12, 16

Q = rng_attention.standard_normal(
    (batch_att, seq_len, dim_att)
)
K = rng_attention.standard_normal(
    (batch_att, seq_len, dim_att)
)
V = rng_attention.standard_normal(
    (batch_att, seq_len, dim_att)
)

scores = (
    np.einsum(
        "bid,bjd->bij",
        Q,
        K,
    )
    / np.sqrt(dim_att)
)

weights = softmax(
    scores,
    axis=-1,
)

attention_output = np.einsum(
    "bij,bjd->bid",
    weights,
    V,
)

padding_mask = np.zeros(
    (seq_len, seq_len),
    dtype=float,
)
padding_mask[:, -3:] = -np.inf

weights_masked = softmax(
    scores + padding_mask,
    axis=-1,
)

output_masked = np.einsum(
    "bij,bjd->bid",
    weights_masked,
    V,
)

print("scores / puntajes:", scores.shape)
print("weights / pesos:", weights_masked.shape)
print("output / salida:", output_masked.shape)
print(
    "Rows sum to 1 / Filas suman 1:",
    np.allclose(
        weights_masked.sum(axis=-1),
        1.0,
    ),
)
print(
    "Largest padded weight / Mayor peso en padding:",
    float(weights_masked[..., -3:].max()),
)

### Interactive attention explorer / Explorador interactivo de atención

Choose:

- example `b`;
- query position `i`;
- mask ON/OFF.

The bar chart shows the attention weights over the 12 key positions.

> 🇪🇸 Elige el ejemplo, la posición query y si la máscara está activa. La gráfica muestra los pesos sobre las 12 posiciones key.

In [ ]:
attention_example = widgets.IntSlider(
    value=0,
    min=0,
    max=batch_att - 1,
    step=1,
    description="Batch b:",
    continuous_update=False,
)

attention_query = widgets.IntSlider(
    value=0,
    min=0,
    max=seq_len - 1,
    step=1,
    description="Query i:",
    continuous_update=False,
)

attention_mask_toggle = widgets.ToggleButtons(
    options=[
        ("Mask ON / Máscara ON", True),
        ("Mask OFF / Máscara OFF", False),
    ],
    value=True,
    description="Padding:",
)

def explore_attention(example, query, use_mask):
    selected = (
        weights_masked[example, query]
        if use_mask
        else weights[example, query]
    )

    fig, ax = plt.subplots(
        figsize=(8.5, 3.6),
        constrained_layout=True,
    )

    bars = ax.bar(
        range(seq_len),
        selected,
    )

    if use_mask:
        for j in range(seq_len - 3, seq_len):
            bars[j].set_hatch("//")

    ax.set_xticks(range(seq_len))
    ax.set_xlabel("key position j / posición key j")
    ax.set_ylabel("attention weight / peso de atención")
    ax.set_title(
        f"Attention weights — b={example}, i={query}\n"
        f"Pesos de atención — máscara={'ON' if use_mask else 'OFF'}"
    )

    plt.show()

    print("Sum / Suma:", float(selected.sum()))
    print(
        "Largest padded weight / Mayor peso padding:",
        float(selected[-3:].max()),
    )
    print("EN: masking before softmax removes padded keys from the weight distribution.")
    print("ES: enmascarar antes de softmax elimina las keys de padding de la distribución de pesos.")

attention_output_widget = widgets.interactive_output(
    explore_attention,
    {
        "example": attention_example,
        "query": attention_query,
        "use_mask": attention_mask_toggle,
    },
)

display(
    widgets.VBox([
        widgets.HBox([
            attention_example,
            attention_query,
        ]),
        attention_mask_toggle,
        attention_output_widget,
    ])
)

## Take-home C — CP versus Tucker / CP frente a Tucker

Notebook 10 used Tucker/HOSVD on a real tensor:

`pickup borough × dropoff borough × hour`

Now compare it with **CP decomposition**.

### Tucker

Tucker uses:

- one factor matrix per mode;
- a separate core tensor.

### CP

CP represents a tensor as a sum of rank-1 components:

`component 1 + component 2 + ...`

Each CP component has:

- one pickup factor vector;
- one dropoff factor vector;
- one hour factor vector.

There is **no Tucker-style core tensor**.

> 🇪🇸 Tucker usa factores por modo y un núcleo. CP expresa el tensor como suma de componentes de rango 1 y no usa un núcleo Tucker separado.

In [ ]:
# TODO C / TAREA C
#
# EN:
# 1. Rebuild the real NYC taxi tensor.
# 2. Fit a rank-3 CP decomposition with TensorLy.
# 3. Reconstruct the tensor and compute relative error.
# 4. Compare its parameter count with Tucker rank (2,2,3).
# 5. Inspect pickup, dropoff, and hour factors for each CP component.
#
# ES:
# 1. Reconstruye el tensor real de taxis de NYC.
# 2. Ajusta una descomposición CP de rango 3 con TensorLy.
# 3. Reconstruye el tensor y calcula el error relativo.
# 4. Compara sus parámetros con Tucker (2,2,3).
# 5. Inspecciona los factores de origen, destino y hora.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

try:
    import tensorly as tl
    from tensorly.decomposition import parafac
except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "tensorly",
        ],
        check=True,
    )
    import tensorly as tl
    from tensorly.decomposition import parafac

TAXIS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/taxis.csv"
)

taxis = pd.read_csv(TAXIS)

taxis["pickup_dt"] = pd.to_datetime(
    taxis["pickup"],
    errors="coerce",
)
taxis["hour"] = taxis["pickup_dt"].dt.hour

sub_taxi = taxis.dropna(
    subset=[
        "pickup_borough",
        "dropoff_borough",
        "hour",
    ]
).copy()

sub_taxi["hour"] = sub_taxi["hour"].astype(int)

pb = sorted(
    sub_taxi["pickup_borough"].unique()
)
db = sorted(
    sub_taxi["dropoff_borough"].unique()
)

p_idx = {
    name: i
    for i, name in enumerate(pb)
}
d_idx = {
    name: i
    for i, name in enumerate(db)
}

T_taxi = np.zeros(
    (
        len(pb),
        len(db),
        24,
    ),
    dtype=float,
)

for (p, d, h), count in sub_taxi.groupby(
    [
        "pickup_borough",
        "dropoff_borough",
        "hour",
    ]
).size().items():
    T_taxi[
        p_idx[p],
        d_idx[d],
        int(h),
    ] = float(count)

rank_cp = 3

cp_weights, cp_factors = parafac(
    tl.tensor(T_taxi),
    rank=rank_cp,
    init="svd",
    random_state=0,
    n_iter_max=500,
    tol=1e-9,
)

F_pickup, F_dropoff, F_hour = cp_factors

cp_recon = tl.cp_to_tensor(
    (
        cp_weights,
        cp_factors,
    )
)

cp_error = (
    np.linalg.norm(cp_recon - T_taxi)
    / np.linalg.norm(T_taxi)
)

cp_params = (
    len(cp_weights)
    + sum(
        factor.size
        for factor in cp_factors
    )
)

tucker_ranks = (2, 2, 3)

tucker_params = int(
    np.prod(tucker_ranks)
    + T_taxi.shape[0] * tucker_ranks[0]
    + T_taxi.shape[1] * tucker_ranks[1]
    + T_taxi.shape[2] * tucker_ranks[2]
)

print("Taxi tensor / Tensor taxis:", T_taxi.shape)
print("CP rank / Rango CP:", rank_cp)
print("CP relative error / Error relativo CP:", f"{cp_error:.4f}")
print("CP parameters / Parámetros CP:", int(cp_params))
print("Tucker (2,2,3) parameters / Parámetros:", tucker_params)

### Interactive CP component explorer / Explorador interactivo de componentes CP

Choose one CP component.

You will see its:

- pickup factor;
- dropoff factor;
- hour factor.

Treat these patterns as **exploratory**, not automatically as named real-world “trip types”.

> 🇪🇸 Elige un componente CP y observa sus factores de origen, destino y hora. Interpreta los patrones con cautela.

In [ ]:
cp_component_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=rank_cp,
    step=1,
    description="Component / Componente:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def show_cp_component(component):
    r = component - 1

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12.5, 3.8),
        constrained_layout=True,
    )

    axes[0].bar(
        range(len(pb)),
        F_pickup[:, r],
    )
    axes[0].set_xticks(range(len(pb)))
    axes[0].set_xticklabels(
        pb,
        rotation=35,
        ha="right",
        fontsize=8,
    )
    axes[0].set_title("Pickup factor / Factor origen")

    axes[1].bar(
        range(len(db)),
        F_dropoff[:, r],
    )
    axes[1].set_xticks(range(len(db)))
    axes[1].set_xticklabels(
        db,
        rotation=35,
        ha="right",
        fontsize=8,
    )
    axes[1].set_title("Dropoff factor / Factor destino")

    axes[2].bar(
        range(24),
        F_hour[:, r],
    )
    axes[2].set_xlabel("hour / hora")
    axes[2].set_title("Hour factor / Factor hora")

    fig.suptitle(
        f"CP component {component} / Componente CP {component}"
    )

    plt.show()

    print(
        "Strongest pickup / Origen de mayor magnitud:",
        pb[int(np.argmax(np.abs(F_pickup[:, r])))],
    )
    print(
        "Strongest dropoff / Destino de mayor magnitud:",
        db[int(np.argmax(np.abs(F_dropoff[:, r])))],
    )
    print(
        "Strongest hour / Hora de mayor magnitud:",
        int(np.argmax(np.abs(F_hour[:, r]))),
    )
    print("EN: factor signs and component scaling are not direct causal labels.")
    print("ES: los signos y escalas de los factores no son etiquetas causales directas.")

cp_output = widgets.interactive_output(
    show_cp_component,
    {"component": cp_component_slider},
)

display(
    widgets.VBox([
        cp_component_slider,
        cp_output,
    ])
)

## Take-home D — Cholesky builds correlation / Cholesky construye correlación

This experiment is **synthetic by design**.

We choose the covariance structure ourselves so we know exactly what changed.

### Start with independent noise

`z ~ N(0, I)`

The three variables are independent.

### Apply Cholesky

If:

`Σ = L Lᵀ`

then:

`x = L z`

has covariance approximately `Σ`.

### Why this matters / Por qué importa

Two portfolios can have the same individual asset volatilities but different total risk if the assets move together differently.

Correlation changes the **joint behaviour**.

> 🇪🇸 Elegimos deliberadamente la covarianza para saber cuál es la verdad del experimento.
>
> Cholesky transforma ruido independiente en variables con la estructura de covarianza deseada.

In [ ]:
# TODO D / TAREA D
#
# EN:
# 1. Build Sigma from volatilities and correlations.
# 2. Compute L = cholesky(Sigma).
# 3. Verify L @ L.T == Sigma.
# 4. Transform independent z into correlated x = L @ z.
# 5. Compare portfolio outcomes with and without cross-correlation.
#
# ES:
# 1. Construye Sigma a partir de volatilidades y correlaciones.
# 2. Calcula L = cholesky(Sigma).
# 3. Verifica L @ L.T == Sigma.
# 4. Transforma z independiente en x = L @ z.
# 5. Compara resultados del portafolio con y sin correlación cruzada.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

vol = np.array([
    0.012,
    0.015,
    0.010,
])

base_corr = np.array([
    [1.00, 0.85, 0.20],
    [0.85, 1.00, 0.20],
    [0.20, 0.20, 1.00],
])

Sigma = (
    np.outer(vol, vol)
    * base_corr
)

L = np.linalg.cholesky(
    Sigma
)

print(
    "L @ L.T == Sigma / L @ L.T == Sigma:",
    np.allclose(
        L @ L.T,
        Sigma,
    ),
)

rng_chol = np.random.default_rng(5)

z = rng_chol.standard_normal(
    (3, 100_000)
)

x_corr = L @ z

print(
    "Covariance error / Error de covarianza:",
    f"{np.linalg.norm(np.cov(x_corr) - Sigma):.6e}",
)

weights_portfolio = np.array([
    0.4,
    0.4,
    0.2,
])

mu = np.array([
    0.00030,
    0.00035,
    0.00020,
])

print("EN: Cholesky reproduces the chosen covariance structure from independent Gaussian noise.")
print("ES: Cholesky reproduce la estructura de covarianza elegida a partir de ruido gaussiano independiente.")

### Interactive correlation-risk explorer / Explorador interactivo de correlación y riesgo

Move the correlation between assets 1 and 2.

Their individual volatilities stay fixed.

The plot compares the simulated **one-day portfolio return** under:

- the selected correlation;
- independence.

> 🇪🇸 Cambia la correlación entre los activos 1 y 2. Las volatilidades individuales permanecen fijas. La gráfica compara el retorno diario del portafolio con correlación y con independencia.

In [ ]:
rho_slider = widgets.FloatSlider(
    value=0.85,
    min=-0.50,
    max=0.95,
    step=0.05,
    description="ρ₁₂:",
    continuous_update=False,
    readout_format=".2f",
)

def explore_correlation_risk(rho12):
    corr_live = np.array([
        [1.00, rho12, 0.20],
        [rho12, 1.00, 0.20],
        [0.20, 0.20, 1.00],
    ])

    eigvals = np.linalg.eigvalsh(
        corr_live
    )

    if eigvals.min() <= 1e-10:
        print("EN: this correlation combination is not positive definite, so Cholesky cannot be used.")
        print("ES: esta combinación de correlaciones no es definida positiva, por lo que Cholesky no puede usarse.")
        print("Smallest eigenvalue / Menor autovalor:", eigvals.min())
        return

    Sigma_live = (
        np.outer(vol, vol)
        * corr_live
    )

    L_live = np.linalg.cholesky(
        Sigma_live
    )

    rng_local = np.random.default_rng(123)

    z_local = rng_local.standard_normal(
        (3, 80_000)
    )

    corr_returns = (
        mu[:, None]
        + L_live @ z_local
    )

    independent_scale = np.diag(vol)

    ind_returns = (
        mu[:, None]
        + independent_scale @ z_local
    )

    portfolio_corr = np.einsum(
        "a,an->n",
        weights_portfolio,
        corr_returns,
    )

    portfolio_ind = np.einsum(
        "a,an->n",
        weights_portfolio,
        ind_returns,
    )

    fig, ax = plt.subplots(
        figsize=(8.5, 3.8),
        constrained_layout=True,
    )

    ax.hist(
        portfolio_ind,
        bins=60,
        density=True,
        alpha=0.55,
        label="independent / independiente",
    )

    ax.hist(
        portfolio_corr,
        bins=60,
        density=True,
        alpha=0.55,
        label=f"ρ₁₂={rho12:.2f}",
    )

    ax.set_xlabel("one-day portfolio return / retorno diario")
    ax.set_ylabel("density / densidad")
    ax.set_title(
        "Dependence changes portfolio risk\n"
        "La dependencia cambia el riesgo"
    )
    ax.legend()

    plt.show()

    print("ρ₁₂:", f"{rho12:.2f}")
    print(
        "Std correlated / Std correlacionado:",
        f"{portfolio_corr.std():.5f}",
    )
    print(
        "Std independent / Std independiente:",
        f"{portfolio_ind.std():.5f}",
    )
    print(
        "5% percentile correlated / Percentil 5% correlacionado:",
        f"{np.percentile(portfolio_corr, 5):.5f}",
    )
    print("EN: covariance changes joint risk even when individual volatilities stay fixed.")
    print("ES: la covarianza cambia el riesgo conjunto aunque las volatilidades individuales permanezcan fijas.")

chol_output = widgets.interactive_output(
    explore_correlation_risk,
    {"rho12": rho_slider},
)

display(
    widgets.VBox([
        rho_slider,
        chol_output,
    ])
)

## Take-home E — Audio denoising by low-rank STFT / Reducción de ruido de audio con STFT de bajo rango

The voice recording is **real** and pinned to a specific file hash.

The added Gaussian noise is **synthetic by design** because a known clean reference allows us to measure SNR objectively.

### Pipeline / Pipeline

**real voice → controlled noise → STFT matrix → SVD truncation → ISTFT → SNR**

### STFT in plain language / STFT en lenguaje sencillo

STFT cuts the audio into short overlapping time windows and asks:

> **Which frequencies are present in each short moment?**

The result is a matrix:

`frequency × time`

### Why SVD? / ¿Por qué SVD?

If useful voice structure is concentrated in dominant singular directions more strongly than noise, a truncated SVD may improve SNR.

But:

> **Low rank does not automatically mean “clean audio”.**

We must measure the result.

> 🇪🇸 STFT transforma el audio en una matriz frecuencia × tiempo. Una SVD truncada puede ayudar si la señal útil está más concentrada que el ruido en las direcciones singulares dominantes, pero debemos medir el resultado.

In [ ]:
# TODO E / TAREA E
#
# EN:
# 1. Download and verify the pinned voice.wav.
# 2. Add controlled Gaussian noise at 5 dB target SNR.
# 3. Compute STFT(noisy) and its complex SVD.
# 4. Try several truncation ranks k.
# 5. Reconstruct with ISTFT and measure SNR for each k.
# 6. Find the best tested k.
# 7. Explain why full rank returns to the noisy STFT.
#
# ES:
# 1. Descarga y verifica voice.wav.
# 2. Agrega ruido gaussiano controlado a SNR objetivo de 5 dB.
# 3. Calcula STFT(noisy) y su SVD compleja.
# 4. Prueba varios rangos k.
# 5. Reconstruye con ISTFT y mide SNR.
# 6. Encuentra el mejor k probado.
# 7. Explica por qué rango completo regresa a la STFT ruidosa.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

VOICE_URL = (
    "https://raw.githubusercontent.com/pdx-cs-sound/wavs/"
    "ed5ebcbbbc2d11f0adddc9b50b78d581c29f738c/"
    "voice.wav"
)

VOICE_SHA256 = (
    "2c4b4d9d5f90715fdbf599869a465d521638f40ca978b186df96f1543a4d67dc"
)

def fetch_verified_wav(
    url,
    expected_sha256,
):
    from scipy.io import wavfile

    raw = urllib.request.urlopen(
        url,
        timeout=30,
    ).read()

    got = hashlib.sha256(
        raw
    ).hexdigest()

    if got != expected_sha256:
        raise ValueError(
            "checksum mismatch: refusing to use unverified audio"
        )

    return wavfile.read(
        io.BytesIO(raw)
    )

fs, clean_i16 = fetch_verified_wav(
    VOICE_URL,
    VOICE_SHA256,
)

clean = (
    clean_i16.astype(np.float64)
    / 32768.0
)

if clean.ndim > 1:
    clean = clean.mean(axis=1)

rng_audio = np.random.default_rng(42)

TARGET_SNR_DB = 5.0

noise = rng_audio.standard_normal(
    clean.shape
)

noise_scale = np.sqrt(
    np.mean(clean**2)
    / (
        np.mean(noise**2)
        * 10 ** (
            TARGET_SNR_DB / 10
        )
    )
)

noisy = (
    clean
    + noise_scale * noise
)

f_audio, t_audio, Z = signal.stft(
    noisy,
    fs=fs,
    nperseg=1024,
    noverlap=512,
)

U_audio, s_audio, Vh_audio = np.linalg.svd(
    Z,
    full_matrices=False,
)

candidates = [
    2,
    5,
    10,
    20,
    40,
    80,
    len(s_audio),
]

candidates = sorted(
    set(
        min(
            int(k),
            len(s_audio),
        )
        for k in candidates
    )
)

reconstructions = {}
snrs = {}
energies = {}

for k in candidates:
    Zk = (
        U_audio[:, :k]
        * s_audio[:k]
    ) @ Vh_audio[:k, :]

    _, rec = signal.istft(
        Zk,
        fs=fs,
        nperseg=1024,
        noverlap=512,
    )

    n = min(
        len(clean),
        len(rec),
    )

    rec = rec[:n]
    reference = clean[:n]

    reconstructions[k] = rec
    snrs[k] = snr_db(
        reference,
        rec,
    )
    energies[k] = (
        np.sum(s_audio[:k] ** 2)
        / np.sum(s_audio**2)
    )

best_k = max(
    candidates,
    key=lambda k: snrs[k],
)

print(
    "Duration / Duración:",
    f"{len(clean) / fs:.3f} s",
)
print(
    "STFT shape / Forma STFT:",
    Z.shape,
)
print(
    "Measured noisy SNR / SNR ruidoso:",
    f"{snr_db(clean, noisy):.2f} dB",
)
print(
    "Best tested rank / Mejor rango probado:",
    best_k,
)
print(
    "Best tested SNR / Mejor SNR probado:",
    f"{snrs[best_k]:.2f} dB",
)
print()
print("EN: full rank reconstructs the noisy STFT, so it cannot remove the noise we deliberately added.")
print("ES: rango completo reconstruye la STFT ruidosa, por lo que no puede eliminar el ruido que agregamos deliberadamente.")

### Interactive audio rank explorer / Explorador interactivo del rango de audio

Choose an SVD rank.

The notebook shows:

- retained singular-value energy;
- reconstructed SNR;
- rank as a fraction of full rank;
- the SNR curve for all tested ranks.

Turn **Audio / Escuchar** on if you want to compare noisy and reconstructed clips.

> 🇪🇸 Elige un rango SVD. Verás energía retenida, SNR reconstruido, fracción del rango completo y la curva de SNR. Activa **Escuchar** para comparar audio.

In [ ]:
audio_rank = widgets.Dropdown(
    options=[
        (
            f"k={k}",
            k,
        )
        for k in candidates
    ],
    value=best_k,
    description="Rank / Rango:",
    style={"description_width": "105px"},
)

play_audio = widgets.Checkbox(
    value=False,
    description="Audio / Escuchar",
    indent=False,
)

def explore_audio_rank(k, listen):
    selected_snr = snrs[k]
    selected_energy = energies[k]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 3.8),
        constrained_layout=True,
    )

    axes[0].plot(
        candidates,
        [snrs[r] for r in candidates],
        marker="o",
    )
    axes[0].scatter(
        [k],
        [selected_snr],
        s=90,
        zorder=5,
    )
    axes[0].axhline(
        snr_db(clean, noisy),
        linestyle="--",
        linewidth=1,
        label="noisy input / entrada ruidosa",
    )
    axes[0].set_xlabel("SVD rank k / rango SVD k")
    axes[0].set_ylabel("SNR (dB)")
    axes[0].set_title("SNR trade-off / Compromiso SNR")
    axes[0].legend(fontsize=8)

    axes[1].plot(
        candidates,
        [100 * energies[r] for r in candidates],
        marker="o",
    )
    axes[1].scatter(
        [k],
        [100 * selected_energy],
        s=90,
        zorder=5,
    )
    axes[1].set_xlabel("SVD rank k / rango SVD k")
    axes[1].set_ylabel("retained energy (%) / energía retenida (%)")
    axes[1].set_title("Singular-value energy / Energía singular")

    plt.show()

    print("Selected rank / Rango seleccionado:", k)
    print(
        "Rank fraction / Fracción del rango:",
        f"{k / len(s_audio):.1%}",
    )
    print(
        "Retained energy / Energía retenida:",
        f"{100 * selected_energy:.1f}%",
    )
    print(
        "Reconstructed SNR / SNR reconstruido:",
        f"{selected_snr:.2f} dB",
    )
    print(
        "Improvement vs noisy / Mejora vs ruidoso:",
        f"{selected_snr - snr_db(clean, noisy):+.2f} dB",
    )

    if listen:
        rec = reconstructions[k]
        n = len(rec)
        reference = clean[:n]
        noisy_local = noisy[:n]

        peak = max(
            np.abs(reference).max(),
            np.abs(noisy_local).max(),
            np.abs(rec).max(),
        )

        print()
        print("Noisy / Ruidoso")
        display(
            Audio(
                noisy_local / peak,
                rate=fs,
            )
        )

        print("Reconstructed / Reconstruido")
        display(
            Audio(
                rec / peak,
                rate=fs,
            )
        )

audio_output = widgets.interactive_output(
    explore_audio_rank,
    {
        "k": audio_rank,
        "listen": play_audio,
    },
)

display(
    widgets.VBox([
        audio_rank,
        play_audio,
        audio_output,
    ])
)

## Final reasoning challenge / Reto final de razonamiento

The five take-homes look different, but each one asks you to protect meaning while transforming data.

Choose one and identify the central risk.

> 🇪🇸 Los cinco ejercicios parecen distintos, pero todos exigen conservar el significado mientras transformamos los datos. Elige uno e identifica el riesgo central.

In [ ]:
takehome_choice = widgets.Dropdown(
    options=[
        ("PCA", "pca"),
        ("Attention / Atención", "attention"),
        ("CP", "cp"),
        ("Cholesky", "cholesky"),
        ("Audio SVD", "audio"),
    ],
    value="pca",
    description="Take-home:",
)

def explain_takehome_risk(choice):
    risks = {
        "pca": (
            "Numerical scale can dominate variance and change what PCA emphasizes.",
            "La escala numérica puede dominar la varianza y cambiar lo que PCA enfatiza.",
        ),
        "attention": (
            "Padding can receive attention unless it is masked before softmax.",
            "El padding puede recibir atención si no se enmascara antes de softmax.",
        ),
        "cp": (
            "Components may be over-interpreted as uniquely meaningful real-world categories.",
            "Los componentes pueden sobreinterpretarse como categorías reales únicas.",
        ),
        "cholesky": (
            "Ignoring covariance can misrepresent joint risk even when marginal volatilities are correct.",
            "Ignorar la covarianza puede representar mal el riesgo conjunto aunque las volatilidades individuales sean correctas.",
        ),
        "audio": (
            "A lower rank can remove useful voice structure as well as noise.",
            "Un rango menor puede eliminar estructura útil de la voz además del ruido.",
        ),
    }

    en, es = risks[choice]
    print("EN:", en)
    print("ES:", es)

final_output = widgets.interactive_output(
    explain_takehome_risk,
    {"choice": takehome_choice},
)

display(
    widgets.VBox([
        takehome_choice,
        final_output,
    ])
)

## What just happened / Qué acaba de pasar

The five take-homes reuse the workshop rather than introducing five unrelated tricks.

### A — PCA

A mathematically optimal variance approximation can still answer the wrong scientific question if feature scales dominate.

### B — Attention

Two contractions, normalization, and masking turn pairwise similarity into a weighted combination.

### C — CP

Another tensor decomposition changes the representation and interpretability trade-off relative to Tucker.

### D — Cholesky

A factorization can be used constructively to impose a chosen covariance structure.

### E — Audio

Low-rank truncation is useful only when the measured approximation improves the signal criterion that matters.

## The sentence to leave with / La frase para llevarte

> **Represent the structure you actually have, approximate only when you can measure the loss, and never let shape manipulation hide what the axes mean.**

> 🇪🇸
>
> **Representa la estructura que realmente tienes, aproxima solo cuando puedes medir la pérdida y nunca permitas que una manipulación de formas oculte el significado de los ejes.**

### Final self-check / Autoevaluación final

When you see a tensor in a new project, ask these questions before writing complicated code:

1. What does each axis mean?
2. Which operations preserve that meaning?
3. Which information will be discarded or approximated?
4. How will I measure whether that loss is acceptable?

> 🇪🇸 Cuando encuentres un tensor en un nuevo proyecto, pregunta primero qué significa cada eje, qué operaciones conservan ese significado, qué información se perderá y cómo medirás si esa pérdida es aceptable.

---

## Done with the workshop / Fin del taller 🎉

That is the whole workshop. Thank you for participating.

> 🇪🇸 Ese es todo el taller. Gracias por participar.
>
> Ya tienes una forma práctica de pensar sobre tensores: **primero el significado de los ejes, después la operación matemática**.

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)